# 04 — Structured Streaming, UDFs & AQE

In [1]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 07:10:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
W20260408 07:10:45.720468   146 MemoryArbitrator.cpp:84] Query memory capacity[288.00MB] is set for NOOP arbitrator which has no capacity enforcement
26/04/08 07:10:46 WARN SparkShimProvider: Spark runtime version 4.0.2 is not matched with Gluten's fully tested version 4.0.1


Spark 4.0.2  |  Mode: Gluten/Velox


26/04/08 07:10:49 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=0], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(id#0L, Some(Etc/UTC)), class name: ToPrettyString.
[Stage 0:>                                                          (0 + 4) / 4]

+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



## Structured Streaming

In [2]:
import time
stream_df = spark.readStream.format('rate').option('rowsPerSecond', 20).load()
agg = stream_df.withColumn('bucket', (F.col('value') % 5).cast('string'))                .groupBy('bucket').count()
query = agg.writeStream.outputMode('complete').format('memory').queryName('rate_agg').start()
time.sleep(5)
spark.sql('SELECT * FROM rate_agg ORDER BY bucket').show()
query.stop()


26/04/08 07:10:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-86e2a30c-e393-4321-9a65-c5eb6f2343bb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/08 07:10:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/08 07:10:52 WARN GlutenFallbackReporter: Validation failed for plan: Exchange, due to: [FallbackByBackendSettings] Validation failed on node Exchange
26/04/08 07:10:52 WARN GlutenFallbackReporter: Validation failed for plan: WriteToDataSourceV2, due to: [FallbackByBackendSettings] Validation failed on node WriteToDataSourceV2
26/04/08 07:10:52 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=2], due to: [FallbackByBackendSettings] Valid

+------+-----+
|bucket|count|
+------+-----+
+------+-----+



26/04/08 07:10:56 WARN GlutenFallbackReporter: Validation failed for plan: TakeOrderedAndProject[QueryId=6], due to: [FallbackByBackendSettings] Validation failed on node TakeOrderedAndProject
26/04/08 07:10:56 WARN DAGScheduler: Failed to cancel job group 8edcddb2-5204-4ce0-9378-526bbbcdc070. Cannot find active jobs for it.
26/04/08 07:10:56 WARN DAGScheduler: Failed to cancel job group 8edcddb2-5204-4ce0-9378-526bbbcdc070. Cannot find active jobs for it.


## UDF & Pandas UDF

In [3]:
import pandas as pd
from pyspark.sql.functions import udf, pandas_udf

@udf(returnType=StringType())
def classify_salary(salary):
    if salary is None: return 'unknown'
    if salary > 90000: return 'high'
    if salary > 70000: return 'mid'
    return 'low'

@pandas_udf(DoubleType())
def tax_estimate(salary: pd.Series) -> pd.Series:
    return salary * 0.3

schema = StructType([StructField('id',IntegerType()),StructField('name',StringType()),
                     StructField('department',StringType()),StructField('salary',DoubleType()),StructField('country',StringType())])
employees = spark.createDataFrame([(1,'Alice','Engineering',95000.0,'US'),(2,'Bob','Marketing',72000.0,'UK'),
    (3,'Carol','Engineering',88000.0,'US'),(4,'Dave','HR',65000.0,'DE')], schema)
employees.withColumn('band', classify_salary('salary'))          .withColumn('tax',  tax_estimate('salary')).show()


26/04/08 07:10:56 WARN GlutenFallbackReporter: Validation failed for plan: BatchEvalPython[QueryId=7], due to: 
 - Validation failed with exception from: EvalPythonExecTransformer, reason: Not supported python udf: classify_salary(salary#33)#35.
26/04/08 07:10:56 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=7], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(id#30, Some(Etc/UTC)), class name: ToPrettyString.
[Stage 6:=======================================>                   (2 + 1) / 3]

+---+-----+-----------+-------+-------+----+-------+
| id| name| department| salary|country|band|    tax|
+---+-----+-----------+-------+-------+----+-------+
|  1|Alice|Engineering|95000.0|     US|high|28500.0|
|  2|  Bob|  Marketing|72000.0|     UK| mid|21600.0|
|  3|Carol|Engineering|88000.0|     US| mid|26400.0|
|  4| Dave|         HR|65000.0|     DE| low|19500.0|
+---+-----+-----------+-------+-------+----+-------+



## Adaptive Query Execution (AQE)

In [4]:
print('AQE enabled:', spark.conf.get('spark.sql.adaptive.enabled'))
skewed = spark.range(200_000)     .withColumn('key', F.when(F.col('id') < 190_000, F.lit(1)).otherwise((F.col('id') % 50).cast('int')))     .withColumn('val', F.rand())
other = spark.range(50).withColumn('key', F.col('id').cast('int')).withColumn('info', F.lit('x'))
joined = skewed.join(other, 'key').groupBy('key').count()
joined.explain(True)


AQE enabled: true
== Parsed Logical Plan ==
'Aggregate ['key], ['key, 'count(1) AS count#69]
+- Project [key#64, id#63L, val#65, id#66L, info#68]
   +- Join Inner, (key#64 = key#67)
      :- Project [id#63L, key#64, rand(-9197827608770071763) AS val#65]
      :  +- Project [id#63L, CASE WHEN (id#63L < cast(190000 as bigint)) THEN 1 ELSE cast((id#63L % cast(50 as bigint)) as int) END AS key#64]
      :     +- Range (0, 200000, step=1, splits=Some(4))
      +- Project [id#66L, key#67, x AS info#68]
         +- Project [id#66L, cast(id#66L as int) AS key#67]
            +- Range (0, 50, step=1, splits=Some(4))

== Analyzed Logical Plan ==
key: int, count: bigint
Aggregate [key#64], [key#64, count(1) AS count#69L]
+- Project [key#64, id#63L, val#65, id#66L, info#68]
   +- Join Inner, (key#64 = key#67)
      :- Project [id#63L, key#64, rand(-9197827608770071763) AS val#65]
      :  +- Project [id#63L, CASE WHEN (id#63L < cast(190000 as bigint)) THEN 1 ELSE cast((id#63L % cast(50 as bigint))